In [3]:
!pip install transformers
!pip install sentencepiece
!pip install torch
 
 
import nltk
import nltk.corpus
import json
import pandas as pd
from transformers import XLMRobertaTokenizer, XLMRobertaModel
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import os
from torchvision.io import read_image
from torch.utils.data import Dataset, DataLoader
from torch import nn
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
 
 
 
 
 
 
def parse_data(file):
    for l in open(file,'r'):
        yield json.loads(l)
data = list(parse_data('./Sarcasm_Headlines_Dataset_v2.json'))
headline_list  = []
sarcasm_check = []
for item in data:
    headline_list.append(item["headline"])
    sarcasm_check.append(item["is_sarcastic"])
df =  pd.DataFrame(data)
df["modul_truth_values"] = df["is_sarcastic"].apply(lambda input: [0,1] if  input  == 1 else [1,0])
print(df["modul_truth_values"])

0        [0, 1]
1        [1, 0]
2        [1, 0]
3        [0, 1]
4        [0, 1]
          ...  
28614    [0, 1]
28615    [0, 1]
28616    [1, 0]
28617    [0, 1]
28618    [0, 1]
Name: modul_truth_values, Length: 28619, dtype: object


In [4]:

tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")
model = XLMRobertaModel.from_pretrained("xlm-roberta-base")
inputs = tokenizer("Hello World!", return_tensors="pt")
outputs = model(**inputs)




Some weights of the model checkpoint at xlm-roberta-base were not used when initializing XLMRobertaModel: ['lm_head.dense.bias', 'lm_head.decoder.weight', 'lm_head.layer_norm.bias', 'lm_head.layer_norm.weight', 'lm_head.bias', 'lm_head.dense.weight']
- This IS expected if you are initializing XLMRobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [5]:
def get_BERT_input(headlines, tokenizer):
  input_ids = []
  attention_masks = []
  encoded_dict = tokenizer.batch_encode_plus(
                      headlines,                     
                      add_special_tokens = True, 
                      max_length = 65,           
                      pad_to_max_length = True,
                      return_attention_mask = True,   # Construct attn. masks.
                      return_tensors = 'pt',     # Return pytorch tensors.
                )
  input_ids, attention_mask = encoded_dict['input_ids'], encoded_dict['attention_mask']
  # Add the encoded sentence to the list.    
  input_ids = encoded_dict['input_ids']
  
  # And its attention mask (simply differentiates padding from non-padding).
  attention_mask = encoded_dict['attention_mask']
  return torch.tensor(input_ids), torch.tensor(attention_mask)
  
class SarcasmDataset(Dataset):
    def __init__(self, df):
        self.headlines = df["headline"]
        self.modul_truth_values = df["modul_truth_values"]
    def __len__(self):
        return len(self.headlines)
 
    def __getitem__(self, idx):
        return self.headlines.iloc[idx],torch.tensor(self.modul_truth_values.iloc[idx])
    
 
#get_BERT_input(["Shahbazis wearing a plaid shirt", "We have 60. students in this class"],tokenizer)
 
 
dataset = SarcasmDataset(df)
train_dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [6]:
from torch.optim import AdamW
class SarcasmModel(nn.Module):
    def __init__(self):
        super(SarcasmModel, self).__init__()
        self.XLM = XLMRobertaModel.from_pretrained("xlm-roberta-base")
        self.hidden_1 = nn.Linear(768, 2)
        
        self.to_delete = 2
        self.softmax  = nn.Softmax(dim=1)
 
    def forward(self, b_input_ids,b_attention_mask):
        bert_output = self.XLM(b_input_ids,b_attention_mask)
        hidden_state = bert_output["last_hidden_state"]
        sentence_vector = torch.mean(hidden_state,dim = 1)
       # print(sentence_vector.shape)
        x = self.hidden_1(sentence_vector)
       # print(x.shape)
       
        probabilties = self.softmax(x)
        #print(probabilties)
        
 
        # print(bert_output)
        # print(b_input_ids.shape)
        # print(b_attention_mask.shape)
        return probabilties
 
model = SarcasmModel()
loss_function = nn.BCELoss().to(device)
optimizer = AdamW(model.parameters(),
                  lr = 2e-5 ,
                  eps = 1e-8
                  )
model = model.to(device)
for i_batch,(b_headlines,b_modul_truth_values) in enumerate(train_dataloader):
  b_input_ids, b_attention_mask = get_BERT_input(b_headlines,tokenizer)
  # print(b_input_ids.shape)
  # print(b_attention_mask.shape)
  optimizer.zero_grad()
  b_prediciton = model(b_input_ids.to(device),b_attention_mask.to(device))
  loss = loss_function(b_prediciton,b_modul_truth_values.float().to(device))
  #print(loss)
  model.zero_grad()
  loss.backward()
  optimizer.step()
  # print(loss)
  batch_size = 32
  if i_batch % 10 == 0:
      iteration = i_batch*batch_size
      print("Iteration:", i_batch*batch_size, "Loss:", loss.data)
      batch_accuracy = torch.mean(torch.sum(b_prediciton * b_modul_truth_values.to(device), dim=1))
      print("Batch Accuracy:", batch_accuracy.data*100)
      if iteration % 5120 == 0:
        # torch.save(model.state_dict(), expt_folder + "SarcasmModel.pt")
        print("Saved Model")




Some weights of the model checkpoint at xlm-roberta-base were not used when initializing XLMRobertaModel: ['lm_head.dense.bias', 'lm_head.decoder.weight', 'lm_head.layer_norm.bias', 'lm_head.layer_norm.weight', 'lm_head.bias', 'lm_head.dense.weight']
- This IS expected if you are initializing XLMRobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you c

Iteration: 0 Loss: tensor(0.6856, device='cuda:0')
Batch Accuracy: tensor(50.5356, device='cuda:0')
Saved Model
Iteration: 320 Loss: tensor(0.6748, device='cuda:0')
Batch Accuracy: tensor(50.9830, device='cuda:0')
Iteration: 640 Loss: tensor(0.6520, device='cuda:0')
Batch Accuracy: tensor(52.4130, device='cuda:0')
Iteration: 960 Loss: tensor(0.6823, device='cuda:0')
Batch Accuracy: tensor(67.1508, device='cuda:0')
Iteration: 1280 Loss: tensor(0.5016, device='cuda:0')
Batch Accuracy: tensor(74.4481, device='cuda:0')
Iteration: 1600 Loss: tensor(0.3988, device='cuda:0')
Batch Accuracy: tensor(70.6037, device='cuda:0')
Iteration: 1920 Loss: tensor(0.2563, device='cuda:0')
Batch Accuracy: tensor(82.1349, device='cuda:0')
Iteration: 2240 Loss: tensor(0.2586, device='cuda:0')
Batch Accuracy: tensor(81.8012, device='cuda:0')
Iteration: 2560 Loss: tensor(0.3293, device='cuda:0')
Batch Accuracy: tensor(80.9731, device='cuda:0')
Iteration: 2880 Loss: tensor(0.4075, device='cuda:0')
Batch Accurac

KeyboardInterrupt: ignored

In [115]:
def predict(b_headline, tokenizer, model):
  b_input_ids, b_attention_mask = get_BERT_input(b_headline, tokenizer)

  b_predictions = model(b_input_ids.to(device), b_attention_mask.to(device))
  print(b_predictions)
  sarcastic_probability = b_predictions.data[0][1].item() * 100
  not_sarcastic_probability = b_predictions.data[0][0].item() * 100
  print_string = "Sarcastic:", f'{sarcastic_probability:.2f}', "Not Sarcastic:", f'{not_sarcastic_probability:.2f}'
  # return b_predictions
  return print_string



predict([""], tokenizer, model)





tensor([[0.9535, 0.0465]], device='cuda:0', grad_fn=<SoftmaxBackward>)


/usr/local/lib/python3.7/dist-packages/transformers/tokenization_utils_base.py:2110: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  FutureWarning,
/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:18: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).


('Sarcastic:', '4.65', 'Not Sarcastic:', '95.35')